In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("lab04.ipynb")

# Lab04: Data Cleaning and EDA

In this lab, you will be working on understanding and visualizing a dataset from the City of Berkeley containing data on calls to the Berkeley Police Department. Information about the dataset can be found [at this link](https://data.cityofberkeley.info/Public-Safety/Berkeley-PD-Calls-for-Service/k2nh-s5h5).

Note: This lab will not work on older versions of Python; make sure to work on DataHub.

<span style="color:red; font-weight:bold">
Warning: This lab includes an analysis of crime in Berkeley.
</span>

---

## Ethical Note:  
- This lab uses real-world crime data for educational purposes.  
- While the dataset is anonymized, it contains references to incidents that may involve violence or other sensitive topics.  
- The purpose of this exercise is to practice data analysis skills while also considering the ethical implications of working with socially sensitive datasets.  
- Please proceed with awareness and respect for the communities represented in the data.


## Setup

Note that we configure a custom default figure size. Virtually every default aspect of matplotlib [can be customized](https://matplotlib.org/users/customizing.html).

In [ ]:
import pandas as pd
import numpy as np
import zipfile
import matplotlib
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (8, 5)

<br/><br/>
<hr style="border: 5px solid #8a8c8c;" />
<hr style="border: 1px solid #ffcd00;" />


## Part 1: Cleaning and Exploring the Data


### Let's now load the CSV file we have into a `pandas.DataFrame` object.

In [ ]:
calls = pd.read_csv("data/Berkeley_PD_-_Calls_for_Service.csv")
calls.head()

We see that the fields include a case number, the offense type, the date and time of the offense, the "CVLEGEND" which appears to be related to the offense type, a "CVDOW" which has no apparent meaning, a date added to the database, and the location spread across four fields.

Let's also check some basic information about these files using the `DataFrame.info` and `DataFrame.describe` methods.

In [ ]:
calls.info()

In [ ]:
calls.describe()

Notice that the functions above reveal type information for the columns, as well as some basic statistics about the numerical columns found in the DataFrame. However, we still need more information about what each column represents. Let's explore the data further in Question 1.

Before we go over the fields to see their meanings, the cell below will verify that all the events happened in Berkeley by grouping on the `City` and `State` columns. You should see that all of our data falls into one group.

In [ ]:
calls.groupby(["City","State"]).count()

<br>

---

### Example 1
Above, when we called `head()` on the Dataframe `calls`, it seemed like `OFFENSE` and `CVLEGEND` both contained information about the type of event reported. What is the difference in meaning between the two columns? One way to probe this is to look at the `value_counts` for each Series.

In [ ]:
calls['OFFENSE'].value_counts().head(10)

In [ ]:
calls['CVLEGEND'].value_counts().head(10)

Above, it seems like `OFFENSE` is more specific than `CVLEGEND`, e.g. "LARCENY" vs. "THEFT FELONY (OVER $950)". If you're unfamiliar with the term, "larceny" is a legal term for theft of personal property.

To get a sense of how many subcategories there are for each `OFFENSE`, we will set `calls_by_cvlegend_and_offense` equal to a multi-indexed series where the data is first indexed on the `CVLEGEND` and then on the `OFFENSE`, and the data is equal to the number of offenses in the database that match the respective `CVLEGEND` and `OFFENSE`. As you can see, `calls_by_cvlegend_and_offense["LARCENY", "THEFT FROM PERSON"]` returns 24 which means there are 24 instances of larceny with offense of type "THEFT FROM PERSON" in the database

In [ ]:
calls_by_cvlegend_and_offense = calls.groupby(["CVLEGEND", "OFFENSE"]).size()
calls_by_cvlegend_and_offense["LARCENY", "THEFT FROM PERSON"]

### Question 1

The examples above show how `CVLEGEND` gives a broad crime category while `OFFENSE` gives a more specific description.

Set `answer1` equal to a **list of strings** containing all unique `OFFENSE` values associated with `CVLEGEND == "BURGLARY - VEHICLE"`. Your answer should not contain duplicates.

You may type the values manually, but a `pandas` expression is recommended.

In [ ]:
answer1 = ...

In [ ]:
grader.check("q1")

<br/><br/>
<hr style="border: 5px solid #8a8c8c;" />
<hr style="border: 1px solid #ffcd00;" />

## Part 2: Visualization


## Pandas Examples

Pandas offers basic functionality for plotting. For example, the `DataFrame` and `Series` classes both have a `plot` method. 

As you learn to do data visualization, you may find the [pandas plotting documentation](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.plot.html)  helpful!

Note, we are going to cover more topics w.r.t. visualization later in this course. 

As an example of the built-in plotting functionality of pandas, the following example uses `plot` method of the `Series` class to generate a `barh` plot type to visually display the value counts for `CVLEGEND`.

There are also many other plots that we will explore throughout the lab.

In [ ]:
ax = calls['CVLEGEND'].value_counts().plot(kind='barh')
ax.set_ylabel("Crime Category")
ax.set_xlabel("Number of Calls")
ax.set_title("Number of Calls By Crime Type");
ax2 = plt.gca()



## An Additional Note on Plotting in Jupyter Notebooks

You may have noticed that many of our code cells involving plotting end with a semicolon (;). This prevents any extra output from the last line of the cell that we may not want to see. Try adding this to your own code in the following questions!

<br>

---

### Question 2

Now it is your turn to make some plots using `pandas`.  Let's start by transforming the data so that it is easier to work with. We then will look at some distributions of the data. 

The CVDOW field isn't named helpfully and it is hard to see the meaning from the data alone. According to the website linked at the top of this notebook, CVDOW is actually indicating the day that events happened. 0->Sunday, 1->Monday ... 6->Saturday. 



<br>

---

#### Question 2a

The `CVDOW` column uses `0` for Sunday through `6` for Saturday. Create a new column named `DayType` in `calls` containing `"Weekend"` for Sunday (`0`) and Saturday (`6`), and `"Weekday"` for Monday through Friday (`1`–`5`).

**Hint:** You can use [Series.map](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.map.html), [Series.isin](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.isin.html) or another vectorized `pandas` approach.

In [ ]:
day_type_map = {0: "Weekend", 1: "Weekday", 2: "Weekday", 3: "Weekday", 4: "Weekday", 5: "Weekday", 6: "Weekend"}
calls["DayType"] = ...
calls[["CVDOW", "DayType"]].head()

In [ ]:
grader.check("q2a")

<br>

---

#### Question 2b

The `EVENTTM` column stores times in `HH:MM` format. Extract the **minute** portion and create a new integer column named `Minute` in `calls`. For example, `"22:17"` should produce the integer `17`.

**Hint:** String methods such as `.str.split()` can be useful. *Your code should only require one line*


In [ ]:
...

In [ ]:
grader.check("q2b")

<br>

---

#### Question 2c

Using `pandas`, compute the number of calls for each `DayType` and store the result in a Series named `daytype_counts`. Then create a **bar plot** of these counts.

Your Series should include both `Weekday` and `Weekend`. Be sure that your plot has labeled axes and a title.

In [ ]:
...

In [ ]:
grader.check("q2c")

<br><br>

---

### Question 3

It seems weekdays generally have slightly more calls than Saturday or Sunday, but the difference does not look significant.  

We can break down into some particular types of events to see their distribution. For example, let's make a bar plot for the CVLEGEND "ROBBERY". Which day is the peak for "ROBBERY"?



<br>

---

#### Question 3a

Filter `calls` to include only rows where `CVLEGEND` is **"LARCENY"** and store the result in `filtered`. Then create a Series named `larceny_by_day` containing the number of LARCENY calls for each `CVDOW`, ordered from `0` through `6`, and make a vertical bar plot of these counts.

In [ ]:

# Auto-graded (9 points)
filtered = ...
larceny_by_day = ...

# Manually graded plot (5 points)
ax_q3a = ...
plt.ylabel(...)
plt.xlabel(...)
plt.title(...)

In [ ]:
grader.check("q3a")

<br>

---

#### Question 3b

Using `larceny_by_day` from Question 3a, determine which numeric day code (`CVDOW`) has the **largest number of LARCENY calls**. Store the integer day code in `answer3b`. If there is a tie, use the smallest day code.

answer3b = ...

In [ ]:
answer3b = ...

In [ ]:
grader.check("q3b")

<br><br>

---

### Question 4

Let's look at when calls were recorded by different types of crime. 





<!-- BEGIN QUESTION -->

<br>

---

### Question 4a

Create a boxplot that examines the **minute within the hour** (`Minute`) for each crime category (`CVLEGEND`). Store the returned axes object in `ax4a`. Rotate the category labels for readability and include a title and axis labels.

To construct this plot use the [DataFrame.boxplot](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.boxplot.html) documentation.

Be sure that your axes are labeled and that your plot is titled. 

In [ ]:
...

<!-- END QUESTION -->

<br>

---

#### Question 4b

Compute the interquartile range of `Minute` for every `CVLEGEND` category. Store these IQR values in a Series named `minute_iqr_by_crime`, then set `answer4b` equal to the crime category with the **largest IQR**. Use `Q3 - Q1`, where `Q1` is the 25th percentile and `Q3` is the 75th percentile.

**Function Hints:** 
[`DataFrame.groupby()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html) · 
[`SeriesGroupBy.quantile()`](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.SeriesGroupBy.quantile.html) · 
[`Series.idxmax()`](https://pandas.pydata.org/docs/reference/api/pandas.Series.idxmax.html)

In [ ]:


q1_minute = ...
q3_minute = ...
minute_iqr_by_crime = ...
answer4b = ...


In [ ]:
grader.check("q4b")

<br><br>

<hr style="border: 5px solid #8a8c8c;" />
<hr style="border: 1px solid #ffcd00;" />

## Congratulations! You have finished Lab 04!


Congrats! You are finished with this assignment.

**Important**: To make sure the test cases run correctly, click `Kernel>Restart & Run All` and make sure all of the test cases are still passing. Doing so will submit your code for you. 

If your test cases are no longer passing after restarting, it's likely because you're missing a variable, or the modifications that you'd previously made to your DataFrame are no longer taking place (perhaps because you deleted a cell). 

You may submit this assignment as many times as you'd like before the deadline.

**You must restart and run all cells before submitting. Otherwise, you may pass test cases locally, but not on our servers. We will not entertain regrade requests of the form, “my code passed all of my local test cases, but failed the autograder”.**

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit. **Please save before exporting!**

In [ ]:
# Save your notebook first, then run this cell to export your submission.
grader.export(pdf=False)